In [1]:
import numpy as np
import pandas as pd

from constants import *
from self_consistency_k import bdg_sc_full_k

In [2]:
initial_seeds = np.array([
    [0,0,0,0],  # normal state
    # [0.1, -0.1, 0, 0],  # px 
    # [0, 0, 0.1, -0.1],  # py
    # [0.1, -0.1, 0.1, -0.1],  # px + py
    [0.1, -0.1, 0.1j, -0.1j],  # px + i*py
    [0.1, 0.1, -0.1, -0.1],  # d-wave
    # [0.1+0.1, 0.1-0.1, -0.1, -0.1],  # d-wave + px
    # [0.1, 0.1, -0.1+0.1, -0.1-0.1],  # d-wave + py
    [0.1, 0.1, 0.1, 0.1],  # s-wave
    # [0.1+0.1, 0.1-0.1, 0.1, 0.1],  # s-wave + px
    # [0.1, 0.1, 0.1+0.1, 0.1-0.1] # s-wave + py
], dtype=np.complex128)
seed_strings = [
    "normal state",
    # "px", "py", "px+py", 
    "px+i*py",
    # "d-wave", "d-wave+px", "d-wave+py",
    "s-wave", 
    "s-wave+px",
    "s-wave+px + py",
    # "s-wave+py"
]

In [3]:
mu_arr = np.linspace(2.11, 2.30, 30)
print(mu_arr)
T_arr = np.linspace(0.001, 0.1, 34)
T_arr = np.linspace(0.001, 0.3, 50)

print(T_arr)

[2.11       2.11655172 2.12310345 2.12965517 2.1362069  2.14275862
 2.14931034 2.15586207 2.16241379 2.16896552 2.17551724 2.18206897
 2.18862069 2.19517241 2.20172414 2.20827586 2.21482759 2.22137931
 2.22793103 2.23448276 2.24103448 2.24758621 2.25413793 2.26068966
 2.26724138 2.2737931  2.28034483 2.28689655 2.29344828 2.3       ]
[0.001      0.00710204 0.01320408 0.01930612 0.02540816 0.0315102
 0.03761224 0.04371429 0.04981633 0.05591837 0.06202041 0.06812245
 0.07422449 0.08032653 0.08642857 0.09253061 0.09863265 0.10473469
 0.11083673 0.11693878 0.12304082 0.12914286 0.1352449  0.14134694
 0.14744898 0.15355102 0.15965306 0.1657551  0.17185714 0.17795918
 0.18406122 0.19016327 0.19626531 0.20236735 0.20846939 0.21457143
 0.22067347 0.22677551 0.23287755 0.23897959 0.24508163 0.25118367
 0.25728571 0.26338776 0.2694898  0.27559184 0.28169388 0.28779592
 0.29389796 0.3       ]


In [4]:
mu_arr = np.linspace(1.8, 4.0, 1)
T_arr = np.linspace(0.001, 0.2, 1)

mu_arr = [2.0457142857]  #works for s + px and s + py
# mu_arr = [1]
T_arr = [0.0001]
free_tol = 0.01
print(mu_arr)
print(T_arr)

[2.0457142857]
[0.0001]


In [5]:
t=1
V_prime=1.5
V = 1.5

h = np.array([0, 0, 0])
Nx, Ny = 75, 75

atol = 1e-5
rtol = 1e-3
maxiter=1000

In [6]:
def stable_config(out, atol=1e-6, rtol=0.1):
    stable = {}
    F_max = out.corr[np.argmax(np.abs(out.corr))]
    if np.abs(F_max) < atol:
        return stable
    for i, c in enumerate(out[1:-3]):
        if np.abs(c) > atol and np.abs(c) > rtol * np.abs(F_max):
            stable[corr_strings[i]] = c
    return stable
def same_stable_dict(d1, d2, atol=1e-6, rtol=1e-3):
    if d1.keys() != d2.keys():
        return False

    for k in d1:
        if not np.isclose(d1[k], d2[k], atol=atol, rtol=rtol):
            return False

    return True

In [7]:
initial_seeds_uu = np.array([
    [0, 0, 0, 0],

    # [0.1, -0.1, 0, 0],
    # [0.1, -0.1, 0.1, -0.1],
    # [0.1, -0.1, 0.1j, -0.1j],

    # [0, 0, 0, 0],
    # [0, 0, 0, 0],
    # [0, 0, 0, 0],

    # [0.1, -0.1, 0, 0],
    # [0.1, -0.1, 0.1, -0.1],
    [0.1, -0.1, 0.1j, -0.1j]
], dtype=np.complex128)

initial_seeds_dd = np.array([
    [0, 0, 0, 0],

    # [0, 0, 0, 0],
    # [0, 0, 0, 0],
    # [0, 0, 0, 0],

    # [0.1, -0.1, 0, 0],
    # [0.1, -0.1, 0.1, -0.1],
    # [0.1, -0.1, 0.1j, -0.1j],

    # [0.1, -0.1, 0, 0],
    # [0.1, -0.1, 0.1, -0.1],
    [0.1, -0.1, 0.1j, -0.1j],
], dtype=np.complex128)

seed_strings = [
    "normal state",
    # "uu:px", "uu:px+py", "uu:px+ipy",
    # "dd:px", "dd:px+py", "dd:px+ipy",
    # "uu+dd:px", "uu+dd:px+py", 
    "uu+dd:px+ipy",
]


In [8]:
print_output = True
records = []

mu = 1
T_arr = [0.35]

for T in T_arr:
    best_free = np.inf
    configs = []
    print("====================================")
    print(f"mu={mu:.2f}, T={T:.4f}")
    print("====================================")
    for seed_uu, seed_dd, seed_str in zip(initial_seeds_uu,
                                            initial_seeds_dd, seed_strings):
        print(f"  Seed: {seed_str}")
        print(f"  Seed uu: {seed_uu}, Seed dd: {seed_dd}")
        out = bdg_sc_full_k(
            t, mu, temperature=T,
            V_prime=V, Nx=Nx, Ny=Ny, h=h,
            atol=atol, rtol=rtol,
            maxiter=maxiter,
            Fuu_init=seed_uu,
            Fdd_init=seed_dd
        )

        print(f"F_onsite: {out.F_onsite}")
        print(f"F_swave: {out.F_swave}, F_dwave: {out.F_dwave}")
        print(f"F_px: {out.F_px}, F_py: {out.F_py}")
        print(f"Free_energy = {out.free_energy}")

        stable = stable_config(out, atol=atol)

        print("------------------------------------------")
        if (out.free_energy < best_free and
                np.abs(out.free_energy - best_free) > free_tol):

            best_free = out.free_energy
            configs = [{
                "stable": stable,
                "free": out.free_energy,
            }]

        elif np.abs(out.free_energy - best_free) <= free_tol:
            if len(stable) != 0 and not any(
                same_stable_dict(c["stable"], stable, atol=atol, rtol=rtol)
                for c in configs
            ):
                configs.append({
                    "stable": stable,
                    "free": out.free_energy,
                })

        print(configs)
        print("------------------------------------------")

    records.append({
        "mu": mu,
        "T": T,
        "best_free": best_free,
        "configs": configs
    })
# Create DataFrame
df = pd.DataFrame(records)

mu=1.00, T=0.3500
  Seed: normal state
  Seed uu: [0.+0.j 0.+0.j 0.+0.j 0.+0.j], Seed dd: [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
F_onsite: 0j
F_swave: 0j, F_dwave: 0j
F_px: 0j, F_py: 0j
Free_energy = -10676.230071737915
------------------------------------------
[{'stable': {}, 'free': np.float64(-10676.230071737915)}]
------------------------------------------
  Seed: uu+dd:px+ipy
  Seed uu: [ 0.1+0.j  -0.1+0.j   0. +0.1j -0. -0.1j], Seed dd: [ 0.1+0.j  -0.1+0.j   0. +0.1j -0. -0.1j]
F_onsite: (-3.144709573705896e-20+9.932429557585703e-20j)
F_swave: (-2.5191000745911e-21-3.978554931835066e-20j), F_dwave: (2.3384385881398988e-20+3.3084859721071156e-20j)
F_px: (-7.912539907369161e-20+1.1216035380524328e-19j), F_py: (1.4640701635801083e-19-3.668244654901045e-20j)
Free_energy = -10700.642717999037
------------------------------------------
[{'stable': {'Fuu_px': np.complex128(0.0358100343484264-2.0269339609117797e-17j), 'Fuu_py': np.complex128(1.6283890773608704e-17+0.0358100343484263j), 'Fdd_px':

In [ ]:
mu_arr = np.linspace(0.001, 0.6, 10)
print(mu_arr)

[0.001      0.06755556 0.13411111 0.20066667 0.26722222 0.33377778
 0.40033333 0.46688889 0.53344444 0.6       ]


In [2]:
import sys
import os
import numpy as np
import pandas as pd

from self_consistency_k import bdg_sc_full_k
from phase_utils import stable_config, same_stable_dict
from phase_utils import flatten_df


initial_seeds_uu = np.array([
    [0, 0, 0, 0],

    [0.1, -0.1, 0, 0],
    [0.1, -0.1, 0.1, -0.1],
    [0.1, -0.1, 0.1j, -0.1j],

    [0, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],

    [0.1, -0.1, 0, 0],
    [0.1, -0.1, 0.1, -0.1],
    [0.1, -0.1, 0.1j, -0.1j]
], dtype=np.complex128)

initial_seeds_dd = np.array([
    [0, 0, 0, 0],

    [0, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],

    [0.1, -0.1, 0, 0],
    [0.1, -0.1, 0.1, -0.1],
    [0.1, -0.1, 0.1j, -0.1j],

    [0.1, -0.1, 0, 0],
    [0.1, -0.1, 0.1, -0.1],
    [0.1, -0.1, 0.1j, -0.1j],
], dtype=np.complex128)

seed_strings = [
    "normal state",
    # "uu:px", "uu:px+py",
    "uu:px+ipy",
    # "dd:px", "dd:px+py",
    "dd:px+ipy",
    # "uu+dd:px", "uu+dd:px+py",
    "uu+dd:px+ipy",
]

mu_arr = np.linspace(0.001, 4.5, 1)
T_arr = np.linspace(0.001, 0.2, 1)

free_tol = 0.1

t = 1
V = 1.5
Nx, Ny = 75, 75

atol = 1e-5
rtol = 1e-2
maxiter = 500

h = np.zeros(3)
h[2] = 0.5 * t


def main():
    idx = 0
    print(f"Running job {idx} out of {len(mu_arr)}")

    records = []

    mu = 3.5

    for T in T_arr:
        best_free = np.inf
        configs = []
        print("====================================")
        print(f"mu={mu:.2f}, T={T:.4f}")
        print("====================================")
        for seed_uu, seed_dd, seed_str in zip(initial_seeds_uu,
                                              initial_seeds_dd, seed_strings):
            print(f"  Seed: {seed_str}")
            print(f"  Seed uu: {seed_uu}, Seed dd: {seed_dd}")
            out = bdg_sc_full_k(
                t, mu, temperature=T,
                V_prime=V, Nx=Nx, Ny=Ny, h=h,
                atol=atol, rtol=rtol,
                maxiter=maxiter,
                Fuu_init=seed_uu,
                Fdd_init=seed_dd
            )

            print(f"F_onsite: {out.F_onsite}")
            print(f"F_swave: {out.F_swave}, F_dwave: {out.F_dwave}")
            print(f"F_px: {out.F_px}, F_py: {out.F_py}")
            print(f"Free_energy = {out.free_energy}")

            stable = stable_config(out, atol=atol)

            print("------------------------------------------")
            if (out.free_energy < best_free and
                    np.abs(out.free_energy - best_free) > free_tol):

                best_free = out.free_energy
                configs = [{
                    "stable": stable,
                    "free": out.free_energy,
                }]

            elif np.abs(out.free_energy - best_free) <= free_tol:
                if len(stable) != 0 and not any(
                    same_stable_dict(c["stable"], stable, atol=atol, rtol=rtol)
                    for c in configs
                ):
                    configs.append({
                        "stable": stable,
                        "free": out.free_energy,
                    })

            print(configs)
            print("------------------------------------------")

        records.append({
            "mu": mu,
            "T": T,
            "best_free": best_free,
            "configs": configs
        })

    # Create DataFrame
    df = pd.DataFrame(records)
    df_flat = flatten_df(df)

    file_idx = idx
    os.makedirs("data/muT_V15_prime_BField", exist_ok=True)
    df_flat.to_json(f"data/muT_V15_prime_BField/results_{file_idx:04d}.json")

main()

Running job 0 out of 1
mu=3.50, T=0.0010
  Seed: normal state
  Seed uu: [0.+0.j 0.+0.j 0.+0.j 0.+0.j], Seed dd: [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
F_onsite: 0j
F_swave: 0j, F_dwave: 0j
F_px: 0j, F_py: 0j
Free_energy = -19921.490288532244
------------------------------------------
[{'stable': {}, 'free': np.float64(-19921.490288532244)}]
------------------------------------------
  Seed: uu:px+ipy
  Seed uu: [ 0.1+0.j -0.1+0.j  0. +0.j  0. +0.j], Seed dd: [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
F_onsite: (-1.6396733429819263e-263+9.63858930681796e-252j)
F_swave: (1.4365562300418916e-263-8.444737339159611e-252j), F_dwave: (1.5160028219075603e-264-8.910379605727114e-253j)
F_px: (-2.397019620399909e-252-4.078209924620114e-264j), F_py: (5.986988356199636e-252+1.0185348669865754e-263j)
Free_energy = -19921.490288539175
------------------------------------------
[{'stable': {}, 'free': np.float64(-19921.490288532244)}]
------------------------------------------
  Seed: dd:px+ipy
  Seed uu: [ 0.1+0.j -0.1+0